In [ ]:
# Updated assignment script - week 4

In [ ]:
# Step 1: Import python libraries

In [ ]:
import numpy as np
import rasterio as rio
import geopandas as gpd
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from shapely.ops import unary_union
from shapely.geometry.polygon import Polygon
from cartopy.feature import ShapelyFeature
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

In [ ]:
# Step 2: Define functions

In [ ]:
def percentile_stretch(img, pmin=0., pmax=100.):
    
#docstring
#'''
# Applies a percentile contrast stretch to a 2D raster image.

# The function rescales pixel values between selected minimum
# and maximum percentiles to improve image contrast for display.

# Parameters
# ----------
# image : numpy.ndarray
  #  Two-dimensional raster image array.

# pmin : float, optional
   # Lower percentile value used for stretching.
  #  Default is 0.

# pmax : float, optional
   # Upper percentile value used for stretching.
  #  Default is 100.

# Returns
# -------
# stretched : numpy.ndarray
#    Contrast-stretched image scaled between 0 and 1.

# Raises
# ------
# ValueError
  #  If percentile values are invalid or if the image
 #   is not two-dimensional.
#'''
    
    # here, we make sure that pmin < pmax, and that they are between 0, 100
    if not 0 <= pmin < pmax <= 100:
        raise ValueError('0 <= pmin < pmax <= 100')
    # here, we make sure that the image is only 2-dimensional
    if not img.ndim == 2:
        raise ValueError('Image can only have two dimensions (row, column)')

    minval = np.percentile(img, pmin)
    maxval = np.percentile(img, pmax)

    stretched = (img - minval) / (maxval - minval)  # stretch the image to 0, 1
    stretched[img < minval] = 0  # set anything less than minval to the new minimum, 0.
    stretched[img > maxval] = 1  # set anything greater than maxval to the new maximum, 1.

    return stretched

In [ ]:
def img_display(img, ax, bands, stretch_args=None, **imshow_args):

#docstring
#'''
# Displays a raster image on a Cartopy map after applying
# a percentile contrast stretch to each image band.

# Parameters
# ----------
# image : numpy.ndarray
   # Multi-band raster image array in the format
  #  [band, row, column].

# ax : matplotlib.axes
  #  Axis object used to display the image.

# bands : list
  #  List of band indices to display as an RGB image.

# transform : cartopy.crs
  #  Coordinate reference system of the raster image.

# extent : list
   # Spatial extent of the raster in the format
  #  [xmin, xmax, ymin, ymax].

# pmin : float, optional
   # Lower percentile used for contrast stretching.
  #  Default is 0.

# pmax : float, optional
   # Upper percentile used for contrast stretching.
  #  Default is 100.

# Returns
# -------
# handle :
  #  Image display handle returned by imshow().

# ax :
 #   Updated matplotlib axis containing the displayed image.
#'''
    
    dispimg = img.copy().astype(np.float32)  # make a copy of the original image,
    # but be sure to cast it as a floating-point image, rather than an integer

    for b in range(img.shape[0]):  # loop over each band, stretching using percentile_stretch()
        if stretch_args is None:  # if stretch_args is None, use the default values for percentile_stretch
            dispimg[b] = percentile_stretch(img[b])
        else:
            dispimg[b] = percentile_stretch(img[b], **stretch_args)

    # next, we transpose the image to re-order the indices
    dispimg = dispimg.transpose([1, 2, 0])

    # finally, we display the image
    handle = ax.imshow(dispimg[:, :, bands], **imshow_args)

    return handle, ax

In [ ]:
# Step 3: Load raster pixels and load spatial information

In [ ]:
with rio.open('data_files/NI_Mosaic.tif') as dataset:
    img = dataset.read()
    xmin, ymin, xmax, ymax = dataset.bounds

In [ ]:
# Step 4: Load vector data

In [ ]:
counties = gpd.read_file("data_files/Counties.shp")
towns = gpd.read_file("data_files/Towns.shp")

In [ ]:
# Step 5: Match coordinate systems

In [ ]:
counties = counties.to_crs(dataset.crs)
towns = towns.to_crs(dataset.crs)

In [ ]:
#Step 6: Create the map figure

In [ ]:
# Create the map figure
map_crs = ccrs.UTM(29)

fig, ax = plt.subplots(
    figsize=(10, 10),
    subplot_kw={"projection": map_crs}
)

# Display satellite image
img_display(
    img,
    ax,
    bands=[2, 1, 0],
    stretch_args={"pmin": 2, "pmax": 98},
    transform=map_crs,
    extent=[xmin, xmax, ymin, ymax]
)

# Add county boundaries
counties.boundary.plot(
    ax=ax,
    edgecolor="red",
    linewidth=1.5,
    transform=map_crs
)

# Add towns and cities
towns.plot(
    ax=ax,
    color="black",
    markersize=20,
    transform=map_crs
)

# Add town labels
for idx, row in towns.iterrows():
    ax.text(
        row.geometry.x,
        row.geometry.y,
        row["TOWN_NAME"],
        fontsize=12,
        color="black",
        transform=map_crs
    )

# Create transparent overlay outside county borders
county_union = unary_union(counties.geometry)

background = Polygon([
    (xmin, ymin),
    (xmin, ymax),
    (xmax, ymax),
    (xmax, ymin)
])

mask = background.difference(county_union)

mask_feature = ShapelyFeature(
    [mask],
    map_crs,
    facecolor="black",
    alpha=0.4
)

ax.add_feature(mask_feature)

# Add gridlines
gridlines = ax.gridlines(
    draw_labels=True,
    linestyle="--",
    linewidth=0.5
)

gridlines.top_labels = False
gridlines.right_labels = False

# Set extent to raster bounds
ax.set_extent([xmin, xmax, ymin, ymax], crs=map_crs)

# Add north arrow
ax.annotate(
    'N',
    xy=(0.95, 0.95),          # arrow tip
    xytext=(0.95, 0.85),      # text position
    arrowprops=dict(
        facecolor='white',
        width=4,
        headwidth=12
    ),
    ha='center',
    va='center',
    fontsize=14,
    color='white',
    xycoords=ax.transAxes
)

# Create custom legend items
county_line = mlines.Line2D(
    [],
    [],
    color='red',
    linewidth=1.5,
    label='County Boundaries'
)

town_marker = mlines.Line2D(
    [],
    [],
    color='black',
    marker='o',
    linestyle='None',
    markersize=6,
    label='Towns and Cities'
)

# Add legend to map
ax.legend(
    handles=[county_line, town_marker],
    loc='lower left'
)

# Add title
plt.title("Northern Ireland Satellite Image with Counties and Towns")

# Save and show
fig.savefig(
    "ni_satellite_map.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()